# CLV 이중축 임베딩 결과 진단

기존 Dunnhumby/H&M 60일 validation CSV·JSON만 읽습니다. 모델 학습이나 test/holdout 평가는 하지 않습니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys

drive.mount('/content/drive')
REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
REVIEWED_REF = 'feat/clv-conditioned-moe'
REPO_DIR = Path('/content/clv-m2-lightgcn-runner-diagnostic')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', REVIEWED_REF, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print('진단 소스:', subprocess.run(['git', 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip())

In [ ]:
from diagnose_clv_dual import diagnose_dual_results

RESULT_ROOT = Path('/content/drive/MyDrive/논문/data')
RESULT_SPECS = {
    'dunnhumby': RESULT_ROOT / 'results_clv_dual_dunnhumby',
    'hm_w60': RESULT_ROOT / 'results_clv_dual_hm_w60',
}

def latest_result(folder):
    json_paths = sorted(folder.glob('clv_dual_*.json'), key=lambda path: path.stat().st_mtime, reverse=True)
    assert json_paths, f'결과 JSON이 없습니다: {folder}'
    json_path = json_paths[0]
    csv_path = json_path.with_suffix('.csv')
    assert csv_path.exists(), f'짝이 되는 CSV가 없습니다: {csv_path}'
    return csv_path, json_path

diagnostics = {}
for dataset, folder in RESULT_SPECS.items():
    csv_path, json_path = latest_result(folder)
    output_dir = folder / 'diagnostics'
    diagnostics[dataset] = diagnose_dual_results(csv_path, json_path, output_dir)
    print(dataset, '진단 완료:', output_dir)

In [ ]:
from IPython.display import display, Image

for dataset, result in diagnostics.items():
    print(f'\n===== {dataset}: 선택점 비교 =====')
    display(result['selected_comparison'])
    print('gate 형태별 유효한 개선점')
    display(result['primary_gate_summary'])
    print('정확도 통과 지점의 동일 lambda 대조군 비교')
    display(result['same_lambda_dominance'])
    print('동일 실효강도 보간 비교')
    display(result['matched_strength'])
    print('정확도 통과 전 구간의 동일 실효강도 보간 비교')
    display(result['matched_curve_dominance'])
    print('대조군과 곡선 교차 구간')
    display(result['crossings'])
    print('선택점 인접 lambda 안정성')
    display(result['neighbor_stability'])
    print('CLV 세그먼트별 M1 대비 변화')
    display(result['segments'])
    print('N/V 축 진단')
    print(result['axis_summary'])
    display(Image(filename=result['paths']['lambda_curve_png']))
    display(Image(filename=result['paths']['strength_curve_png']))
    print('한계:', result['limitations']['reason'])
    print('저장 파일:')
    for label, path in result['paths'].items():
        print(f' - {label}: {path}')